# MD set (48 proteins): feature ranges

Per-protein feature table and summary statistics. The prediction error is not used.

In [ ]:
import json, os
import numpy as np, pandas as pd

pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

FIG4 = "../../data/analysis/Fig4"
AF   = f"{FIG4}/all_features"
PRED = f"{FIG4}/results/predictions_N_seed_ensemble_10seed.json"
GT   = "../../data/rBMRB/cluster_30p/PANAV/test.json"
MD_1US = "../../data/md_1us"  # 1-us MD trajectories, one frame per ns
OUT_CSV = f"{FIG4}/md_protein_run48_profile.csv"

## Enumerate the 48 MD entries

In [ ]:
run48 = sorted(d for d in os.listdir(MD_1US)
               if os.path.isdir(os.path.join(MD_1US, d)) and not d.startswith(('_', 'frames_', 'dynamics')))
print(f"n run entries: {len(run48)}")
print(run48)

## Per-protein profile

In [ ]:
with open(PRED) as f: pred_data = json.load(f)
with open(GT)   as f: gt_data   = json.load(f)
prr = json.load(open(f"{AF}/per_residue_records_full.json"))

# 10-seed ensemble: use the already-averaged per-residue "mean" (paired with "mask"), not the
# per-seed "pred" matrix -- same as residue_ss_sasa_error_median_panav_10seedens.ipynb.
pred_dict, mask_dict = pred_data["mean"], pred_data["mask"]
dictionary = {v: k for k, v in gt_data["entry_ID"].items()}

rows, missing, length_mismatch = [], [], 0
for entry in run48:
    key = dictionary.get(entry)
    if key is None or entry not in pred_dict or entry not in prr:
        missing.append(entry); continue

    gt_seq, gt_shifts = gt_data["sequence"][key], gt_data["N"][key]
    n = len(gt_seq)
    pred_arr_full = np.array(pred_dict[entry])
    pred_mask_full = np.array(mask_dict[entry], dtype=bool)
    # predictions are right-padded to the batch's longest sequence -- truncate to n instead of
    # skipping the whole protein when longer; only skip if actually shorter than gt_seq
    if len(pred_arr_full) < n or len(pred_mask_full) < n:
        length_mismatch += 1; continue
    pred_mask = pred_mask_full[:n]

    # proline / None exclusion + the prediction's own mask (tag/flag/outlier exclusion already
    # folded in there) -- no separate 4sigma-BMRB gate on the GT values
    gt_mask_1 = np.array([res != "P" for res in gt_seq])
    none_mask = np.array([g is not None for g in gt_shifts])
    mask = gt_mask_1 & none_mask & pred_mask
    if mask.sum() == 0:
        continue

    resnums = np.where(mask)[0] + 1
    rec_by = {r["resnum"]: r for r in prr[entry]}
    ss     = pd.Series([rec_by.get(int(rn), {}).get("ss_class") for rn in resnums])
    plddt  = [rec_by.get(int(rn), {}).get("plddt") for rn in resnums]
    s2     = [rec_by.get(int(rn), {}).get("s2") for rn in resnums]
    ph     = prr[entry][0].get("ph") if prr[entry] else None

    rows.append({
        "entry": entry, "plen": n, "n_obs": int(mask.sum()),
        "sheet_pct": 100 * (ss == "sheet").mean() if len(ss) else np.nan,
        "helix_pct": 100 * (ss == "helix").mean() if len(ss) else np.nan,
        "plddt_med": float(np.nanmedian([x for x in plddt if x is not None]))
                      if any(x is not None for x in plddt) else np.nan,
        "s2_cov": 100 * np.mean([x is not None for x in s2]) if len(s2) else np.nan,
        "ph": ph,
    })

R = pd.DataFrame(rows)
R["cov_frac"] = R.n_obs / R.plen
print(f"profiled {len(R)} / {len(run48)} run entries (missing: {missing}, length_mismatch: {length_mismatch})")
R.to_csv(OUT_CSV, index=False)
print(f"saved -> {OUT_CSV}")
R.round(2)

## Summary across the 48

In [ ]:
cols = ["plen", "n_obs", "cov_frac", "sheet_pct", "helix_pct", "plddt_med", "s2_cov", "ph"]
summary = R[cols].agg(["min", "max", "median", "mean", "std",
                        lambda s: s.quantile(0.05), lambda s: s.quantile(0.95)]).T
summary.columns = ["min", "max", "median", "mean", "std", "p05", "p95"]
print(summary.round(2).to_string())

print("\nMethods-ready sentence:")
print(
    f"MD trajectories were generated for {len(R)} proteins from the SLAE-CS test set spanning "
    f"{R.plen.min():.0f}-{R.plen.max():.0f} residues (median {R.plen.median():.0f}), each with "
    f"near-complete experimental backbone-15N coverage ({R.n_obs.min():.0f}-{R.n_obs.max():.0f} "
    f"observed residues per protein, {100*R.cov_frac.min():.0f}-{100*R.cov_frac.max():.0f}% of the "
    f"chain), mixed secondary-structure composition ({R.sheet_pct.min():.0f}-{R.sheet_pct.max():.0f}% "
    f"beta-sheet, {R.helix_pct.min():.0f}-{R.helix_pct.max():.0f}% alpha-helix), high-confidence AF2 "
    f"starting structures (per-protein median pLDDT {R.plddt_med.min():.1f}-{R.plddt_med.max():.1f}), "
    f"complete residue-level S2 order-parameter coverage, and experimental pH "
    f"{R.ph.min():.1f}-{R.ph.max():.1f}."
)